## Dependencies

In [1]:
import warnings
warnings.simplefilter("ignore", UserWarning)

In [2]:
import sys
sys.path.append('../')

%load_ext autoreload
%autoreload 2

from libs import ios
from libs import vis
from libs import constants
from libs.bibs import aps
from libs.utils import characternames_handler

## Setup

In [3]:
VERSION = 3
DIR_EVAL = f'../../results/v{VERSION}/gender/predictions_ch/best_models/eval_best_results_<MODEL>.csv'
DIR_PRED = f'../../results/v{VERSION}/gender/predictions_ch/best_models/rs<RS>/'

FN_APS_DEMOGRAPHICS = '/data/datasets/APS/aps_data_processing/shares/aps-2022_25-10-01_e59319256d553035c40188457bb16edbfe02da04.zip' #aps_20240130_2e52fdd7260ea462878821948a2a463ed9acb58a.zip'

MODELS = ['regression', 'classification']

In [4]:
vis.sns_paper_style()

## Data

## Our Predictions & GT

In [5]:
df_gender = ios.pd.DataFrame()
for model in MODELS:
    
    # best overall per model type
    path = DIR_EVAL.replace('<MODEL>', model)
    df_summary = ios.load_csv(path, index_col=0).query("best_overall == True").iloc[0]
    rs = df_summary.rs
    
    # GT
    df_gt = ios.load_csv(ios.path_join(ios.os.path.dirname(df_summary.fn), constants.FN_GT_TRAN_TEST_SAMPLE))
    cols_gt = [c for c in df_gt.columns if c in ['givenname', 'ngender_mean', 'gender_mean', 'ml_sample']]
    df_gt = df_gt[cols_gt].set_index('givenname')
    df_gt.rename(columns={'ngender_mean': 'gt', 'gender_mean': 'gt'}, errors='ignore', inplace=True)

    # all predicitons (all names)
    kind = '_reg_' if model == 'regression' else '_class_'
    path = DIR_PRED.replace('<RS>', str(rs))
    pattern = f'*{kind}*.csv'
    files = ios.listdir(path, pattern, 'csv')
    
    if len(files) == 0:
        print(f'No files found for {model} in {path}')
        continue

    fn = files[0]
    tmp = ios.load_csv(fn)
    assert tmp.shape[0] == tmp.givenname.nunique()
    
    # add gt info
    tmp = tmp.set_index('givenname')
    tmp = tmp.join(df_gt, how='left')
    print(f'{model.title()} | all names with gender: {tmp.shape} | names in gt: {tmp['gt'].notna().sum()}')

    # final df
    cols = [c for c in tmp.columns if c in ['pred_code', 'pred_label', 'pred', 'gt', 'ml_sample']]
    df_gender = tmp[cols].copy() if df_gender.empty else df_gender.join(tmp[cols[:-2]], how='inner')

    
print(df_gender.shape)

[INFO] Found 1 files in ../../results/v3/gender/predictions_ch/best_models/rs2/.
Regression | all names with gender: (46144, 11) | names in gt: 166
[INFO] Found 1 files in ../../results/v3/gender/predictions_ch/best_models/rs2/.
Classification | all names with gender: (46144, 11) | names in gt: 166
(46144, 5)


In [34]:

df_gender.loc[:, 'py_givenname'] = df_gender.apply(lambda row: ''.join(characternames_handler.pinyin_givename(row.name)), axis=1)
df_gender.head(2)

,pred,gt,ml_sample,pred_code,pred_label,py_givenname
givenname,,,,,,
㑇,0.347726,NaN,NaN,0,male,zhou
㑊,0.347726,NaN,NaN,0,male,yi


In [35]:
df_gt = df_gender.query("gt.notna()").copy()
df_gt.sample(2)

,pred,gt,ml_sample,pred_code,pred_label,py_givenname
givenname,,,,,,
冬艳,0.820166,0.833333,test,2,female,dongyan
玉芯,0.806994,0.791667,train,2,female,yuxin


## APS

In [8]:
_df_aps_authors = ios.read_csv_from_zip(FN_APS_DEMOGRAPHICS, 'data/authors.csv')
_df_aps_authors_names = ios.read_csv_from_zip(FN_APS_DEMOGRAPHICS, 'data/author_names.csv')
_df_aps_genders = ios.read_csv_from_zip(FN_APS_DEMOGRAPHICS, 'data/genders.csv')

df_aps = ios.pd.merge(_df_aps_authors, _df_aps_authors_names, how='inner', on='id_author', suffixes=("_a","_n"))
df_aps = ios.pd.merge(df_aps, _df_aps_genders, how='left', on='id_gender')
df_aps = df_aps[['id_author','id_author_name','name','gender','gender_nq', 'disambiguated']] # NOTE: it contains disambiguated True and False

print(f"APS authors: {_df_aps_authors.shape}")
print(f"APS authors names: {_df_aps_authors_names.shape}")
print(f"APS genders: {_df_aps_genders.shape}")
print(f"APS all authors merged: {df_aps.shape}")

APS authors: (874556, 4)
APS authors names: (948771, 4)
APS genders: (3, 2)
APS all authors merged: (948771, 6)


In [9]:
df_aps_clean = aps.clean_aps_names(df_aps)
df_aps_clean_ch_py = aps.detect_pinyin_and_chinese_names(df_aps_clean)
df_aps_clean_ch_py.shape

(874556, 21)

In [10]:
df_aps_clean_ch_py.head(2)

,id_author,alternative_names,n_alternative_names,genders,genders_nq,gender_clean,gender_nq_clean,longest_name,longest_name_clean,givenname_clean,...,is_pinyin_givenname,is_pinyin_familyname,is_pinyin,ch_name,ch_splitname,ch_familyname,ch_givenname,ch_translated_name_in_pinyin,ch_translated_familyname_in_pinyin,ch_translated_givenname_in_pinyin
0,0,[S. Megtert],1,[unknown],[-],unknown,unknown,S. Megtert,s. megtert,s.,...,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,1,"[Aleksa Bjeli, A. Bjeli]",2,"[male, male]","[-, -]",male,unknown,Aleksa Bjeli,aleksa bjeli,aleksa,...,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Comparison

In [61]:
', '.join(df_aps_clean_ch_py.query("ch_givenname.notna()").sample(20).sort_values('ch_givenname').ch_givenname.unique())

'二军, 仁新, 兆华, 兴涛, 创业, 均仁, 巍, 建成, 敏良, 昕, 松, 林港, 林透, 炜平, 玲, 立杰, 聪博, 阳平, 雷, 龙'

In [36]:
df_aps_gt = df_aps_clean_ch_py.query("is_pinyin == True and ch_name.notna() and ch_givenname in @df_gt.index")[['givenname_clean', 'ch_givenname', 'gender_clean', 'gender_nq_clean']].copy()
df_aps_gt.shape

(5, 4)

In [37]:
df_aps_gt

,givenname_clean,ch_givenname,gender_clean,gender_nq_clean
1696,qiong,琼,male,unknown
20603,yaping,亚平,male,unknown
25636,dong,东,male,unknown
37026,fan,帆,male,unknown
67113,yuan,园,male,gm


In [40]:
df_comparison = df_gender.join(df_aps_gt.set_index('ch_givenname').rename(columns={'gender_clean': 'karimi', 'gender_nq_clean': 'burkik'}).drop(columns=['givenname_clean']), how='inner')
df_comparison.loc[:, ['barolo','ours']] = [None, None]
df_comparison

,pred,gt,ml_sample,pred_code,pred_label,py_givenname,karimi,burkik,barolo,ours
东,0.272237,0.166667,train,0,male,dong,male,unknown,None,None
园,0.619605,0.625000,train,1,neutral,yuan,male,gm,None,None
帆,0.376853,0.458333,train,1,neutral,fan,male,unknown,None,None
琼,0.679497,0.812500,train,2,female,qiong,male,unknown,None,None
亚平,0.450442,0.395833,train,1,neutral,yaping,male,unknown,None,None
